In [1]:
import time 
import torch
import numpy as np
import os
from models import ChebGCN, SimpleGCN
from preprocess_multi import preprocess_variable, edges, create_data, dataloaders
from train_multi import slide_window_gcn

inference_start = time.time()
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU mem: {gpu_mem:.1f} GB")
else:
    print("Kjører på CPU")
 
# ── Konfigurasjon — tilpass til din kjøring ───────────────────────────────────
PATH             = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/results_temporal_coll_with_O/weightedCheb/WeightedChebGCN_lr0.001_ep100_lp0.05_K3_mse_grad_edge_weightTrue_use_deltaTrue_ar4'
DATA_PATH        = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc'
VARIABLES        = ['psi', 'q']
WINDOW_LENGTH    = 4
BATCH_SIZE       = 32
SPIN_UP          = 30
FORECAST_HORIZON = 4
MODEL_NAME       = 'ChebGCN'
K                = 3
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
# ── Data ──────────────────────────────────────────────────────────────────────
X_train, y_train, X_test, y_test, X_val, y_val, metadata = preprocess_variable(
    data_path=DATA_PATH, variable=VARIABLES,
    window_length=WINDOW_LENGTH, spin_up=SPIN_UP
)
 
edge_index = edges(metadata['Y'], metadata['X'])
test_graph = create_data(X_test, y_test, edge_index)
_, test_loader, _ = dataloaders(test_graph, test_graph, test_graph, batch_size=BATCH_SIZE)
 
num_out       = y_train.shape[2]
lev           = num_out
window_length = X_train.shape[2] // num_out
 
# ── Last inn modell ───────────────────────────────────────────────────────────
if MODEL_NAME == 'ChebGCN':
    model = ChebGCN(input_channel=X_train.shape[2], hidden_layers=64,
                    output_channel=num_out, K=K, normalization='sym')
else:
    model = SimpleGCN(input_channel=X_train.shape[2], hidden_layers=64,
                      output_channel=num_out)
 
model.load_state_dict(torch.load(os.path.join(PATH, 'best_model.pt'), map_location=device))
model = model.to(device)
model.eval()
 
# ── Test-evaluering med alle steg ─────────────────────────────────────────────
all_predictions = []   # shape per batch: (B, forecast_horizon, num_nodes, channels)
all_targets     = []   # shape per batch: (B, num_nodes, channels)
 
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        x_c   = batch.x.clone()
        steps  = []
        t_steps = []
 
        for step in range(FORECAST_HORIZON):
            if MODEL_NAME == 'ChebGCN':
                pred = model(x_c, batch.edge_index, lambda_max=1.526)
            else:
                pred = model(x_c, batch.edge_index)
            steps.append(pred.cpu())
            t_steps.append(batch.y.cpu())
            x_c = slide_window_gcn(x_c, pred, window_length, lev)
 
        # Stack steg: (forecast_horizon, B, num_nodes, channels) → (B, forecast_horizon, num_nodes, channels)
        batch_preds = torch.stack(steps, dim=1)   
        batch_t = torch.stack(t_steps, dim = 1)     
        all_predictions.append(batch_preds)
        all_targets.append(batch.y.cpu())
 
final_preds   = torch.cat(all_predictions, dim=0).numpy()  # (N, forecast_horizon, num_nodes, channels)
final_targets = torch.cat(all_targets,     dim=0).numpy()  # (N, num_nodes, channels)
 
np.save(os.path.join(PATH, 'predictions_all_steps.npy'), final_preds)
np.save(os.path.join(PATH, 'targets.npy'),               final_targets)
 
print(f'predictions_all_steps shape: {final_preds.shape}')
print(f'targets shape:               {final_targets.shape}')
print('Ferdig!')

inference_time = time.time() - inference_start
print(f"Forecast-tid: {inference_time:.2f} sekunder for {final_preds.shape[0]} samples")


/Users/malenekarlsen/Documents/Master/Vår26/MachineLearning/Advanced_ML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kjører på CPU


/Users/malenekarlsen/Documents/Master/Vår26/MachineLearning/Advanced_ML/Mallis/multi_variable_TPE/preprocess_multi.py:35: FutureWarning: In a future version, xarray will not decode the variable 'time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  data = xr.open_dataset(data_path)


Train runs: (210,), Test runs: (45,), Val runs: (45,)

── Processing variable: psi ──
The training shape of psi: (210, 56, 2, 64, 64)
The test shape of psi: (45, 56, 2, 64, 64)
The validation shape of psi: (45, 56, 2, 64, 64)

── Processing variable: q ──
The training shape of q: (210, 56, 2, 64, 64)
The test shape of q: (45, 56, 2, 64, 64)
The validation shape of q: (45, 56, 2, 64, 64)

The final shapes used for training:
X_train: torch.Size([10290, 4096, 16]), y_train: torch.Size([10290, 4096, 16])
X_test: torch.Size([2205, 4096, 16]), y_test: torch.Size([2205, 4096, 16])
X_val: torch.Size([2205, 4096, 16]), y_val: torch.Size([2205, 4096, 16])

Channel in y: (n_vars_lev = 4):
t+1 : channels 0 - 3
t+2 : channels 4 - 7
t+3 : channels 8 - 11
t+4 : channels 12 - 15


RuntimeError: Error(s) in loading state_dict for ChebGCN:
	size mismatch for conv3.bias: copying a param with shape torch.Size([4]) from checkpoint, the shape in current model is torch.Size([16]).
	size mismatch for conv3.lins.0.weight: copying a param with shape torch.Size([4, 64]) from checkpoint, the shape in current model is torch.Size([16, 64]).
	size mismatch for conv3.lins.1.weight: copying a param with shape torch.Size([4, 64]) from checkpoint, the shape in current model is torch.Size([16, 64]).
	size mismatch for conv3.lins.2.weight: copying a param with shape torch.Size([4, 64]) from checkpoint, the shape in current model is torch.Size([16, 64]).

In [ ]:
#one step for chebgcn
import time 
import torch
import numpy as np
import os
from models import ChebGCN, SimpleGCN
from preprocess_multi import preprocess_variable, edges, create_data, dataloaders
from train_multi import slide_window_gcn

inference_start = time.time()
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU mem: {gpu_mem:.1f} GB")
else:
    print("Kjører på CPU")
 
# ── Konfigurasjon — tilpass til din kjøring ───────────────────────────────────
PATH             = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/results_outside_ofhub/multi_variable/GCN_lr0.001_ep100_lp0.01_K2'
DATA_PATH        = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc'
# Same innstillinger som treningen
VARIABLES     = ['psi', 'q']
WINDOW_LENGTH = 4
BATCH_SIZE    = 32
SPIN_UP       = 20
FORECAST_HORIZON = 1
K = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Last inn data
X_train, y_train, X_test, y_test, X_val, y_val, metadata = preprocess_variable(
    data_path='/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc',
    variable=VARIABLES, window_length=WINDOW_LENGTH, spin_up=SPIN_UP
)

edge_index = edges(metadata['Y'], metadata['X'])
test_graph = create_data(X_test, y_test, edge_index)
_, test_loader, _ = dataloaders(test_graph, test_graph, test_graph, batch_size=BATCH_SIZE)

# Last inn modellen
num_out = y_train.shape[2]
lev = num_out
window_length = X_train.shape[2] // num_out

model = ChebGCN(input_channel=X_train.shape[2], hidden_layers=64,
                output_channel=num_out, K=K, normalization='sym')
model.load_state_dict(torch.load(os.path.join(PATH, 'best_model.pt'), map_location=device))
model = model.to(device)
model.eval()

# Kjør test-evaluering
predictions, targets_list = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        x_c = batch.x.clone()
        for step in range(FORECAST_HORIZON):
            pred = model(x_c, batch.edge_index, lambda_max=1.526)
            x_c = slide_window_gcn(x_c, pred, window_length, lev)
        predictions.append(pred.cpu())
        targets_list.append(batch.y.cpu())

np.save(os.path.join(PATH, 'predictions.npy'), torch.cat(predictions).numpy())
np.save(os.path.join(PATH, 'targets.npy'),     torch.cat(targets_list).numpy())
print('Ferdig! predictions.npy og targets.npy lagret.')

inference_time = time.time() - inference_start
print(f"Forecast-tid: {inference_time:.2f} sekunder for {final_preds.shape[0]} samples")

Kjører på CPU
Train runs: (210,), Test runs: (45,), Val runs: (45,)

── Processing variable: psi ──
  Train shape: (210, 66, 2, 64, 64)
  Test shape:  (45, 66, 2, 64, 64)
  Val shape:   (45, 66, 2, 64, 64)


/Users/malenekarlsen/Documents/Master/Vår26/MachineLearning/Advanced_ML/Mallis/multi_variable/preprocess_multi.py:37: FutureWarning: In a future version, xarray will not decode the variable 'time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  data = xr.open_dataset(data_path)



── Processing variable: q ──
  Train shape: (210, 66, 2, 64, 64)
  Test shape:  (45, 66, 2, 64, 64)
  Val shape:   (45, 66, 2, 64, 64)

Final shapes:
  X_train: torch.Size([13020, 4096, 16]), y_train: torch.Size([13020, 4096, 4])
  X_test:  torch.Size([2790, 4096, 16]),  y_test:  torch.Size([2790, 4096, 4])
  X_val:   torch.Size([2790, 4096, 16]),   y_val:   torch.Size([2790, 4096, 4])


RuntimeError: Error(s) in loading state_dict for ChebGCN:
	Missing key(s) in state_dict: "conv1.lins.0.weight", "conv1.lins.1.weight", "conv1.lins.2.weight", "conv2.lins.0.weight", "conv2.lins.1.weight", "conv2.lins.2.weight", "conv3.lins.0.weight", "conv3.lins.1.weight", "conv3.lins.2.weight". 
	Unexpected key(s) in state_dict: "conv1.lin.weight", "conv2.lin.weight", "conv3.lin.weight". 

In [6]:
#one step for gcn
import time 
import torch
import numpy as np
import os
from models import ChebGCN, SimpleGCN
from preprocess_multi import preprocess_variable, edges, create_data, dataloaders
from train_multi import slide_window_gcn

inference_start = time.time()
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU mem: {gpu_mem:.1f} GB")
else:
    print("Kjører på CPU")
 
# ── Konfigurasjon — tilpass til din kjøring ───────────────────────────────────
PATH             = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/results_outside_ofhub/multi_variable/GCN_lr0.001_ep100_lp0.01_K2'
DATA_PATH        = '/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc'
# Same innstillinger som treningen
VARIABLES     = ['psi', 'q']
WINDOW_LENGTH = 4
BATCH_SIZE    = 32
SPIN_UP       = 20
FORECAST_HORIZON = 1
K = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Last inn data
X_train, y_train, X_test, y_test, X_val, y_val, metadata = preprocess_variable(
    data_path='/Users/malenekarlsen/documents/Master/Vår26/MachineLearning/Advanced_ML/train.nc',
    variable=VARIABLES, window_length=WINDOW_LENGTH, spin_up=SPIN_UP
)

edge_index = edges(metadata['Y'], metadata['X'])
test_graph = create_data(X_test, y_test, edge_index)
_, test_loader, _ = dataloaders(test_graph, test_graph, test_graph, batch_size=BATCH_SIZE)

# Last inn modellen
num_out = y_train.shape[2]
lev = num_out
window_length = X_train.shape[2] // num_out

model = model = SimpleGCN(input_channel=X_train.shape[2], hidden_layers=64, output_channel=num_out)
model.load_state_dict(torch.load(os.path.join(PATH, 'best_model.pt'), map_location=device))
model = model.to(device)
model.eval()

# Kjør test-evaluering
predictions, targets_list = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        x_c = batch.x.clone()
        for step in range(FORECAST_HORIZON):
            pred = model(x_c, batch.edge_index)
            x_c = slide_window_gcn(x_c, pred, window_length, lev)
        predictions.append(pred.cpu())
        targets_list.append(batch.y.cpu())

np.save(os.path.join(PATH, 'predictions.npy'), torch.cat(predictions).numpy())
np.save(os.path.join(PATH, 'targets.npy'),     torch.cat(targets_list).numpy())
print('Ferdig! predictions.npy og targets.npy lagret.')

inference_time = time.time() - inference_start
final_preds = torch.cat(predictions)
print(f"Forecast-tid: {inference_time:.2f} sekunder for {final_preds.shape[0]} samples")

Kjører på CPU
Train runs: (210,), Test runs: (45,), Val runs: (45,)

── Processing variable: psi ──
  Train shape: (210, 66, 2, 64, 64)
  Test shape:  (45, 66, 2, 64, 64)
  Val shape:   (45, 66, 2, 64, 64)


/Users/malenekarlsen/Documents/Master/Vår26/MachineLearning/Advanced_ML/Mallis/multi_variable/preprocess_multi.py:37: FutureWarning: In a future version, xarray will not decode the variable 'time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  data = xr.open_dataset(data_path)



── Processing variable: q ──
  Train shape: (210, 66, 2, 64, 64)
  Test shape:  (45, 66, 2, 64, 64)
  Val shape:   (45, 66, 2, 64, 64)

Final shapes:
  X_train: torch.Size([13020, 4096, 16]), y_train: torch.Size([13020, 4096, 4])
  X_test:  torch.Size([2790, 4096, 16]),  y_test:  torch.Size([2790, 4096, 4])
  X_val:   torch.Size([2790, 4096, 16]),   y_val:   torch.Size([2790, 4096, 4])
Ferdig! predictions.npy og targets.npy lagret.
Forecast-tid: 100.06 sekunder for 11427840 samples
